In [ ]:
# testcomment 03:58 24 05 2026
import time, math
import numpy as np
import PIL.Image
from pynq import Overlay, MMIO, allocate, Clocks
import svo_builder

# SET CLOCK FREQ (def = 100)
Clocks.fclk0_mhz = 100

BITSTREAM = '/home/xilinx/jupyter_notebooks/svo_system.bit'
IMG_W, IMG_H = 320, 240
BYTES_PER_PIXEL = 4   # 32-bit XRGB

def to_q16(f): return int(f * 65536) & 0xFFFF_FFFF

def pack_rgb(r, g, b): return ((int(r)&0xFF)<<16) | ((int(g)&0xFF)<<8) | (int(b)&0xFF)

def normalise(v):
    l = math.sqrt(sum(x**2 for x in v))
    return [x/l for x in v] if l > 1e-9 else v
def cross(a, b):
    return [a[1]*b[2]-a[2]*b[1], a[2]*b[0]-a[0]*b[2], a[0]*b[1]-a[1]*b[0]]

# ---------------------------------------------------------------------------
# Load bitstream
# ---------------------------------------------------------------------------
print('Loading bitstream ...')
ol = Overlay(BITSTREAM)
ip = ol.top_0

VDMA_BASE = ol.ip_dict['axi_vdma_0']['phys_addr']
vdma = MMIO(VDMA_BASE, 0x1000)

HSIZE  = IMG_W * BYTES_PER_PIXEL
STRIDE = IMG_W * BYTES_PER_PIXEL

frame_buf  = allocate(shape=(IMG_H, IMG_W, BYTES_PER_PIXEL), dtype=np.uint8)
frame_phys = frame_buf.physical_address

# Reset S2MM and wait for self-clear
vdma.write(0x30, 0x4)
while vdma.read(0x30) & 0x4: pass

vdma.write(0xAC, frame_phys)
vdma.write(0xB0, frame_phys)
vdma.write(0xB4, frame_phys)
vdma.write(0xA8, STRIDE)
vdma.write(0xA4, HSIZE)
vdma.write(0x30, 0x3)
vdma.write(0xA0, IMG_H)

s2mm_sr = vdma.read(0x34)
print(f'VDMA S2MM ready: phys=0x{frame_phys:08X}, status=0x{s2mm_sr:08X}')
if s2mm_sr & 0x1:
    print('WARNING: S2MM is Halted — check HSIZE/VSIZE/addresses and re-run')

ModuleNotFoundError: No module named 'pynq'

In [ ]:
# ---------------------------------------------------------------------------
# Build and upload SVO
# ---------------------------------------------------------------------------
print('Building SVO ...')
grid  = svo_builder.build_world()
root  = svo_builder.build_svo(grid)
nodes = svo_builder.flatten_svo(root)
words = svo_builder.serialise_nodes(nodes)
print(f'  {len(nodes)} nodes -> {len(words)} words ({len(words)*4} bytes)')

print('Uploading SVO to BRAM ...')
ip.write(0x48, 0)
for w in words:
    ip.write(0x4C, w)
print('  Upload done.')

In [ ]:
# ---------------------------------------------------------------------------
# Sky colour and camera setup
# ---------------------------------------------------------------------------
SKY_R, SKY_G, SKY_B = 135, 206, 235
ip.write(0x68, pack_rgb(SKY_R, SKY_G, SKY_B))

pos   = [32.0, 40.0, -20.0]
fwd   = normalise([32.0 - pos[0], 4.0 - pos[1], 32.0 - pos[2]])
right = normalise(cross(fwd, [0, 1, 0]))
up    = cross(right, fwd)

fov_scale = math.tan(math.radians(60) / 2) / (IMG_W / 2)

ip.write(0x08, to_q16(pos[0]))
ip.write(0x0C, to_q16(pos[1]))
ip.write(0x10, to_q16(pos[2]))
ip.write(0x14, to_q16(right[0]))
ip.write(0x18, to_q16(right[1]))
ip.write(0x1C, to_q16(right[2]))
ip.write(0x20, to_q16(up[0]))
ip.write(0x24, to_q16(up[1]))
ip.write(0x28, to_q16(up[2]))
ip.write(0x2C, to_q16(fwd[0]))
ip.write(0x30, to_q16(fwd[1]))
ip.write(0x34, to_q16(fwd[2]))
ip.write(0x38, to_q16(fov_scale))

print('Camera:')
print(f'  pos   = {[round(x,3) for x in pos]}')
print(f'  fwd   = {[round(x,3) for x in fwd]}')
print(f'  right = {[round(x,3) for x in right]}')
print(f'  up    = {[round(x,3) for x in up]}')
print(f'  scale = {fov_scale:.6f}')

In [ ]:
# ---------------------------------------------------------------------------
# Helper functions — run this cell once, then use do_render() anywhere
# ---------------------------------------------------------------------------
FSM_STATES = {
    0:'S_IDLE', 1:'S_RAY_SETUP', 2:'S_ROOT_SLAB', 3:'S_ENTER_NODE',
    4:'S_BRAM_WAIT', 5:'S_CHECK_CHILD', 6:'S_EMPTY', 7:'S_SOLID',
    8:'S_MIXED', 9:'S_POP_STACK', 10:'S_MISS', 11:'S_WAIT_SHADE',
    12:'S_WRITE_PIXEL', 13:'S_NEXT_PIXEL'
}

def decode_dbg(dbg, pxpy):
    state   = dbg & 0xF
    tvalid  = (dbg >> 4) & 1
    tready  = (dbg >> 5) & 1
    rs_wait = (dbg >> 6) & 0x1F   # 5-bit field — bits[10:6]
    px      = (pxpy >> 8) & 0x1FF
    py      = pxpy & 0xFF
    return state, tvalid, tready, rs_wait, px, py

def arm_vdma():
    """Reset and re-arm the VDMA S2MM channel. Must be called before every render."""
    vdma.write(0x30, 0x4)
    while vdma.read(0x30) & 0x4: pass
    vdma.write(0xAC, frame_phys)
    vdma.write(0xB0, frame_phys)
    vdma.write(0xB4, frame_phys)
    vdma.write(0xA8, STRIDE)
    vdma.write(0xA4, HSIZE)
    vdma.write(0x30, 0x3)
    vdma.write(0xA0, IMG_H)
    s2mm_sr = vdma.read(0x34)
    if s2mm_sr & 0x1:
        print(f'WARNING: S2MM still halted after re-arm (status=0x{s2mm_sr:08X})')

def do_render(timeout=15.0, verbose=True):
    """
    Arm VDMA, trigger one render, wait for completion.
    Returns (frame_rgb, elapsed, completed, final_dbg).
      frame_rgb  : (240,320,3) uint8 RGB array, or None if trigger failed
      elapsed    : wall-clock seconds
      completed  : True if all 76800 pixels were written (no unwritten R=0 pixels)
      final_dbg  : (state, tvalid, tready, rs_wait, px, py) snapshot at render end
    """
    arm_vdma()

    dbg0, pxpy0 = ip.read(0x78), ip.read(0x7C)
    st0, tv0, tr0, rw0, px0, py0 = decode_dbg(dbg0, pxpy0)
    if verbose:
        print(f'Pre-trigger: status=0x{ip.read(0x04):08X}  '
              f'state={st0}({FSM_STATES.get(st0,"?")})  rs_wait={rw0}  '
              f'tvalid={tv0}  tready={tr0}  px={px0}  py={py0}')

    t0 = time.time()
    ip.write(0x00, 1)

    # Wait for busy=1
    while not (ip.read(0x04) & 0x1):
        if time.time() - t0 > 0.5:
            dbg, pxpy = ip.read(0x78), ip.read(0x7C)
            st, tv, tr, rw, px, py = decode_dbg(dbg, pxpy)
            print(f'ERROR: trigger did not fire — '
                  f'state={st}({FSM_STATES.get(st,"?")})  px={px}  py={py}  '
                  f'dbg_raw=0x{dbg:08X}')
            return None, time.time() - t0, False, (st, tv, tr, rw, px, py)

    # Poll until busy=0
    last_print = t0
    while ip.read(0x04) & 0x1:
        now = time.time()
        if verbose and now - last_print >= 1.0:
            dbg, pxpy = ip.read(0x78), ip.read(0x7C)
            st, tv, tr, rw, px, py = decode_dbg(dbg, pxpy)
            print(f'  t={now-t0:.1f}s  state={st}({FSM_STATES.get(st,"?")})  '
                  f'px={px}  py={py}  tvalid={tv}  tready={tr}  rs_wait={rw}')
            last_print = now
        if time.time() - t0 > timeout:
            print(f'TIMEOUT after {timeout:.0f}s — render hung')
            break
        time.sleep(0.001)

    elapsed = time.time() - t0

    time.sleep(0.05)   # let VDMA finish DMA to DDR
    dbg, pxpy = ip.read(0x78), ip.read(0x7C)
    final = decode_dbg(dbg, pxpy)
    st, tv, tr, rw, px, py = final

    frame_buf.invalidate()
    frame_rgb = np.array(frame_buf[:, :, [2, 1, 0]])

    # Pixels never written stay at 0; sky=135 and white=255 are both non-zero
    zero_px  = int((frame_rgb[:, :, 0] == 0).sum())
    white_px = int(np.sum(np.all(frame_rgb == 255, axis=2)))
    completed = (zero_px == 0)

    vdma_sr = vdma.read(0x34)

    if verbose:
        if completed:
            print(f'Render COMPLETE in {elapsed:.3f}s ({1/elapsed:.2f} FPS)  white={white_px}')
        else:
            print(f'Render INCOMPLETE in {elapsed:.3f}s — '
                  f'stopped near px={px}  py={py}  '
                  f'state={st}({FSM_STATES.get(st,"?")})  tvalid={tv}  tready={tr}')
            print(f'  Pixels never written (R=0): {zero_px}   white={white_px}')
            if vdma_sr & 0x70:
                print(f'  VDMA error bits set: status=0x{vdma_sr:08X}')

    return frame_rgb, elapsed, completed, final

print('Helper functions loaded: arm_vdma(), do_render()')

In [ ]:
# ---------------------------------------------------------------------------
# HDMI display: arm the VDMA MM2S (read) channel to continuously scan the
# rendered DDR frame out to the HDMI monitor (VDMA MM2S -> axis_upscale_2x 2x
# -> v_axi4s_vid_out -> rgb2dvi -> HDMI OUT, 640x480@60). Render path untouched;
# this is additive. Run AFTER a render cell. Re-rendering updates the screen live.
# ---------------------------------------------------------------------------
from pynq import MMIO
import time

vdma_base = ol.ip_dict['axi_vdma_0']['phys_addr']   # same VDMA as S2MM
mm2s = MMIO(vdma_base, 0x100)                        # MM2S register bank

IMG_W, IMG_H = 320, 240
STRIDE = IMG_W * 4    # 1280 bytes/line
HSIZE  = IMG_W * 4    # 1280
VSIZE  = IMG_H        # 240

MM2S_VDMACR, MM2S_VDMASR        = 0x00, 0x04
MM2S_START_A1, MM2S_FRMDLY_STRIDE = 0x5C, 0x58
MM2S_HSIZE, MM2S_VSIZE          = 0x54, 0x50

mm2s.write(MM2S_VDMACR, 0x4)                         # reset, wait self-clear
while mm2s.read(MM2S_VDMACR) & 0x4:
    pass
for off in (MM2S_START_A1, MM2S_START_A1 + 4, MM2S_START_A1 + 8):
    mm2s.write(off, frame_phys)                      # all 3 fstores -> rendered frame (park)
mm2s.write(MM2S_FRMDLY_STRIDE, STRIDE)
mm2s.write(MM2S_HSIZE, HSIZE)
mm2s.write(MM2S_VDMACR, 0x1)                         # RS=1, park on frame 0
mm2s.write(MM2S_VSIZE, VSIZE)                        # write VSIZE LAST -> arms the channel
time.sleep(0.05)

sr = mm2s.read(MM2S_VDMASR)
print(f"MM2S status = {sr:#010x}  (halted={sr & 1}, err[6:4]={(sr >> 4) & 0x7})")
assert (sr >> 4) & 0x7 == 0, "VDMA MM2S error flags set"
print("HDMI armed - rendered frame should now be on the monitor (640x480, 2x upscaled).")


In [ ]:
import math

# ─── Shading register setup (Phase 2) ────────────────────────────────────────
# Must be called once before every render trigger.

def pack_rgb(r, g, b):
    return ((int(r) & 0xFF) << 16) | ((int(g) & 0xFF) << 8) | (int(b) & 0xFF)

# Light direction: normalised (1, 2, 1.5)
_lm = math.sqrt(1.0**2 + 2.0**2 + 1.5**2)
ld = (1.0/_lm, 2.0/_lm, 1.5/_lm)
ol.top_0.write(0x3C, to_q16(ld[0]))   # light_dir_x
ol.top_0.write(0x40, to_q16(ld[1]))   # light_dir_y
ol.top_0.write(0x44, to_q16(ld[2]))   # light_dir_z

# Color LUT (block_id 0..5)
lut_colors = [
    (0,   0,   0),    # 0 air (unreachable)
    (120, 120, 120),  # 1 stone
    (60,  160,  40),  # 2 grass
    (255,   0,   0),  # 3 glowing
    (0,   0,   0),    # 4 unused
    (0,   0,   0),    # 5 unused
]
for i, (r, g, b) in enumerate(lut_colors):
    ol.top_0.write(0x50 + i * 4, pack_rgb(r, g, b))

# Sky and fog colours
ol.top_0.write(0x68, pack_rgb(135, 206, 235))   # sky colour
ol.top_0.write(0x6C, pack_rgb(180, 200, 220))   # fog colour

# Fog start distance and shadow bias (Q16.16)
ol.top_0.write(0x70, to_q16(15.0))   # fog_start
ol.top_0.write(0x74, to_q16(0.5))    # shadow_bias

print("Shading registers written.")

In [ ]:
# ---------------------------------------------------------------------------
# Single render
# ---------------------------------------------------------------------------
frame_rgb, elapsed, completed, final_dbg = do_render()
if frame_rgb is not None:
    image = PIL.Image.fromarray(frame_rgb, 'RGB')
    display(image)

In [ ]:
# ---------------------------------------------------------------------------
# DIAGNOSTIC: Run N renders and compare for non-determinism
#
# Answers two questions:
#   1. Are the wrong pixels the same every run? (deterministic = logic bug)
#   2. Do renders ever stop early / hang?       (Step 2 hang capture)
#
# VERDICT:
#   All renders identical  -> deterministic errors -> logic bug
#   Every render differs   -> non-deterministic    -> timing issue
#   Mixed                  -> both
# ---------------------------------------------------------------------------
N_RENDERS = 5

renders  = []
metadata = []

print(f'Running {N_RENDERS} renders for consistency check...')
print('=' * 60)

for i in range(N_RENDERS):
    print(f'\n--- Render {i+1}/{N_RENDERS} ---')
    frame, elapsed, completed, dbg = do_render(verbose=True)
    if frame is not None:
        renders.append(frame)
        metadata.append({'elapsed': elapsed, 'completed': completed, 'dbg': dbg})

# ---------------------------------------------------------------------------
# Analysis
# ---------------------------------------------------------------------------
print('\n' + '=' * 60)
print(f'SUMMARY: {len(renders)}/{N_RENDERS} renders captured')

n_complete = sum(m['completed'] for m in metadata)
print(f'  Completed fully: {n_complete}/{len(metadata)}')
for i, m in enumerate(metadata):
    if not m['completed']:
        st, tv, tr, rw, px, py = m['dbg']
        print(f'  Render {i+1} incomplete: '
              f'state={st}({FSM_STATES.get(st,"?")})  px={px}  py={py}  '
              f'tvalid={tv}  tready={tr}')

print()
if len(renders) >= 2:
    diff_counts = []
    for i in range(1, len(renders)):
        diff = np.any(renders[0] != renders[i], axis=2)
        n = int(diff.sum())
        diff_counts.append(n)
        print(f'  Render 1 vs render {i+1}: {n:,} pixels differ')

    max_diff = max(diff_counts)
    min_diff = min(diff_counts)

    print()
    if max_diff == 0:
        print('>>> VERDICT: ALL RENDERS IDENTICAL')
        print('    Errors are DETERMINISTIC — logic bug, not timing')
    elif min_diff == 0:
        print('>>> VERDICT: SOME RENDERS IDENTICAL, SOME DIFFER')
        print('    Mix of deterministic (logic) and non-deterministic (timing) errors')
    else:
        print('>>> VERDICT: EVERY RENDER DIFFERS FROM RENDER 1')
        print('    Errors are NON-DETERMINISTIC — timing issue confirmed')

    always_diff = np.ones((IMG_H, IMG_W), dtype=bool)
    any_diff    = np.zeros((IMG_H, IMG_W), dtype=bool)
    for r in renders[1:]:
        d = np.any(renders[0] != r, axis=2)
        always_diff &= d
        any_diff    |= d
    print()
    print(f'  Pixels wrong in EVERY comparison (deterministic component): {int(always_diff.sum()):,}')
    print(f'  Pixels that ever differ across runs (full non-det scope):    {int(any_diff.sum()):,}')

    # Print coordinates of always-wrong pixels (up to 20) for sim follow-up
    n_always = int(always_diff.sum())
    if 0 < n_always <= 200:
        coords = np.argwhere(always_diff)
        print(f'\n  Always-wrong pixel coords (py, px) — add these to sim testbench:')
        for row, col in coords[:20]:
            r1 = tuple(int(x) for x in renders[0][row, col])
            r2 = tuple(int(x) for x in renders[1][row, col])
            print(f'    py={row:3d}  px={col:3d}  render1={r1}  render2={r2}')
        if n_always > 20:
            print(f'    ... and {n_always - 20} more')

# Reference comparison (generate on PC with sim/gen_reference.py, scp to PYNQ)
ref_path = '/home/xilinx/jupyter_notebooks/reference_render.npy'
try:
    ref = np.load(ref_path)
    print(f'\nReference render loaded from {ref_path}:')
    for i, r in enumerate(renders):
        wrong        = np.any(r != ref, axis=2)
        false_hits   = int(np.sum(np.all(r   == [255,255,255], axis=2) &
                                  ~np.all(ref == [255,255,255], axis=2)))
        false_misses = int(np.sum(~np.all(r   == [255,255,255], axis=2) &
                                   np.all(ref == [255,255,255], axis=2)))
        print(f'  Render {i+1}: {int(wrong.sum()):,} wrong  '
              f'({false_hits} false hits,  {false_misses} false misses)')
except FileNotFoundError:
    print(f'\n(No reference_render.npy at {ref_path})')
    print( '  Generate with: python sim/gen_reference.py  then scp the .npy here')

for i, r in enumerate(renders):
    path = f'/home/xilinx/jupyter_notebooks/render_{i+1}.png'
    PIL.Image.fromarray(r, 'RGB').save(path)
print(f'\nSaved render_1.png ... render_{len(renders)}.png')

In [ ]:
# Display a specific render (change index 0..N-1)
PIL.Image.fromarray(renders[0], 'RGB')

In [ ]:
# Optionally save the single render
image.save('/home/xilinx/jupyter_notebooks/render.png')
print('Saved to render.png')

In [ ]:
# Stop VDMA and free frame buffer
vdma.write(0x30, 0x0)
frame_buf.freebuffer()